# Tutorial 1: MLP Search on MNIST

Demonstrates global (NSGA-II) + local (QAT + pruning) search for MLP architectures targeting FPGA deployment.

All parameters are controlled via `t1_config.yaml` in this directory.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

import yaml
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils.tf_global_search import GlobalSearchTF
from utils.tf_local_search_separated import local_search_entrypoint
from utils.tf_data_preprocessing import load_and_preprocess_mnist

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
tf.get_logger().setLevel('ERROR')
print("TensorFlow Version:", tf.__version__)

cfg = yaml.safe_load(open(Path.cwd() / "t1_config.yaml"))
ds_cfg = cfg["dataset"]
s_cfg = cfg["search"]
ss_cfg = cfg["search_space"]
ls_cfg = cfg["local_search"]
out_cfg = cfg["output"]

RESULTS_DIR = out_cfg["results_dir"]
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results dir: {RESULTS_DIR}")
print(f"n_trials={s_cfg['n_trials']}, epochs={s_cfg['epochs']}, subset_size={ds_cfg['subset_size']}")

## Dataset: MNIST

In [ ]:
x_train_viz, y_train_viz, _, _ = load_and_preprocess_mnist(
    resize_val=ds_cfg["resize_val"],
    subset_size=ds_cfg["subset_size"],
    flatten=False,
    one_hot=False,
)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_viz[i].squeeze(), cmap='gray')
    plt.title(f"Label: {y_train_viz[i]}")
    plt.axis('off')
plt.suptitle("Sample MNIST Images")
plt.tight_layout()
plt.show()

## Global Search: Finding the Best Architectural Trade-offs

Runs NSGA-II multi-objective optimization over MLP architectures, optimizing accuracy and BOPs.

In [ ]:
obj_names = s_cfg["objective_names"]
max_flags = s_cfg["maximize_flags"]

searcher = GlobalSearchTF(
    search_space_path=ss_cfg,
    results_dir=RESULTS_DIR,
)

study = searcher.run_search(
    model_type=s_cfg["model_type"],
    n_trials=s_cfg["n_trials"],
    epochs=s_cfg["epochs"],
    dataset=ds_cfg["name"],
    subset_size=ds_cfg["subset_size"],
    resize_val=ds_cfg["resize_val"],
    objectives=obj_names,
    maximize_flags=max_flags,
    use_hardware_metrics=s_cfg["use_hardware_metrics"],
)

print("Global Search Complete!")

## Analyzing the Global Search Results

In [ ]:
results_df = pd.DataFrame(searcher.results)
if not results_df.empty:
    best = results_df.loc[results_df['performance_metric'].idxmax()]
    print(f"Best Trial: {best['trial']}  Accuracy: {best['performance_metric']:.4f}  BOPs: {best['bops']:.2e}")
    print(f"Pareto front plots saved to: {RESULTS_DIR}")
else:
    print("No results to visualize.")

## Local Search: Compressing the Best Model

Applies Quantization-Aware Training (QAT) and iterative magnitude pruning to the best architecture found.

In [ ]:
LOCAL_RESULTS_DIR = os.path.join(RESULTS_DIR, "local_search_separated")
LOCAL_CONFIG_PATH = os.path.join(RESULTS_DIR, "local_search_config.yaml")
ARCH_YAML_PATH = os.path.join(RESULTS_DIR, "best_model_for_local_search.yaml")

local_search_settings = {
    "pruning_settings": {
        "iterations": ls_cfg["pruning_iterations"],
        "epochs_per_iteration": ls_cfg["pruning_epochs"],
        "pruning_rate": ls_cfg["pruning_rate"],
    },
    "qat_settings": {
        "epochs": ls_cfg["qat_epochs"],
        "precision_pairs": ls_cfg["precision_pairs"],
    },
}
with open(LOCAL_CONFIG_PATH, "w") as f:
    yaml.dump(local_search_settings, f)

x_train, y_train, x_val, y_val = load_and_preprocess_mnist(
    resize_val=ds_cfg["resize_val"],
    subset_size=ds_cfg["subset_size"],
    flatten=True,
    one_hot=True,
)

if os.path.exists(ARCH_YAML_PATH):
    pruning_df, qat_df = local_search_entrypoint(
        architecture_yaml_path=ARCH_YAML_PATH,
        local_search_config_path=LOCAL_CONFIG_PATH,
        dataset=(x_train, y_train, x_val, y_val),
        results_dir=LOCAL_RESULTS_DIR,
    )
else:
    print(f"ERROR: Architecture YAML not found: {ARCH_YAML_PATH}")
    pruning_df, qat_df = pd.DataFrame(), pd.DataFrame()

In [ ]:
if not pruning_df.empty:
    plt.figure(figsize=(10, 6))
    plt.plot(pruning_df['Sparsity'], pruning_df['Accuracy'], marker='o', linewidth=2)
    plt.title('Pruning: Accuracy vs. Sparsity')
    plt.xlabel('Sparsity')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

if not qat_df.empty:
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(qat_df)), qat_df['Accuracy'], tick_label=qat_df['Precision'])
    plt.title('QAT: Accuracy vs. Precision')
    plt.xlabel('Precision')
    plt.ylabel('Accuracy')
    plt.tight_layout()
    plt.show()